# CMIE Household Survey: Final Analysis

This notebook transitions from Data Engineering to Advanced Econometric Analysis on the cleaned `master_panel_stage8.parquet`.

It covers:
1. **Summary Statistics & Sanity Checks**: Validating asset participation against external benchmarks (e.g., Badarinza AIDIS).
2. **Consumption Smoothing (MPC & PIH)**: Testing the marginal propensity to consume, robust to extreme outliers and decomposing into permanent vs transitory income.
3. **Determinants of Co-Holding**: Exploring the puzzle of holding expensive debt and illiquid savings simultaneously, adjusting for selection bias and mechanical artifacts.
4. **Intention-Behavior Gap**: Evaluating if formal workers have a true behavioral advantage in saving (Gold) or just a mechanical one (PF).


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt

# Single source of truth for paths
data_path = os.path.join("Cleaned_Output", "master_panel.parquet")
print(f"Loading {data_path}...")

# Load only necessary columns to save memory
columns_to_load = [
    'HH_ID', 'WAVE_NO', 'MONTH', 'STATE_WAVE', 'REGION_TYPE_WAVE', 'EDU_GROUP_WAVE',
    'TOT_INC', 'TOT_EXP', 'R_HH_WGT_MS_INC', 'R_HH_WGT_W',
    'HAS_SAVING_IN_REAL_ESTATE', 'HAS_SAVING_IN_GOLD', 'HAS_SAVING_IN_PF', 'HAS_SAVING_IN_MF',
    'HIGH_COST_DEBT_FLAG', 'INC_CV', 'COHOLD_FLAG',
    'WILL_SAVE_IN_PF', 'WILL_SAVE_IN_GOLD', 'WILL_SAVE_IN_REAL_ESTATE', 'WILL_SAVE_IN_MF',
    'HAS_SAVED_IN_REAL_ESTATE', 'HAS_SAVED_IN_GOLD', 'HAS_SAVED_IN_PF', 'HAS_SAVED_IN_MF',
    'HAS_FORMAL_EMPLOYED_MEM', 'AGE_YRS_HOH', 'N_MEMBERS',
    # Include Gap columns generated in Stage 7
    'GAP_PF_JanApr_MayAug', 'GAP_GOLD_JanApr_MayAug'
]

# Ensure we don't crash if a gap col is missing from earlier stage runs
schema_cols = pq.ParquetFile(data_path).schema.names
columns_to_load = [c for c in columns_to_load if c in schema_cols]

df = pd.read_parquet(data_path, columns=columns_to_load)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns safely!")


## 1. Data Prep: Panel Rule and Unit Checks
**Why this step?** 
1. We must decide whether to use a balanced panel (only households in all waves) or pooled/unbalanced (all households). Unbalanced is preferred to prevent survivorship bias.
2. We must ensure `TOT_EXP` is in the same monthly units as `TOT_INC`. CMIE sometimes reports expenditure weekly, which creates an artificially low MPC if not adjusted.


In [ ]:
# Decide Panel Rule
wave_counts = df.groupby('HH_ID')['WAVE_NO'].nunique()
df_balanced = df[df['HH_ID'].isin(wave_counts[wave_counts == 3].index)].copy()

PANEL_CHOICE = "unbalanced"
analysis_df = df_balanced.copy() if PANEL_CHOICE == "balanced" else df.copy()
print(f"Using '{PANEL_CHOICE}' panel: {len(analysis_df):,} rows")

# Unit Check for Expenditure
mean_inc = analysis_df['TOT_INC'].mean()
mean_exp = analysis_df['TOT_EXP'].mean()
ratio = mean_exp / mean_inc

WEEKS_PER_MONTH = 52 / 12

if ratio < 0.5:
    print(f"WARNING: mean(EXP)/mean(INC) = {ratio:.3f}. Adjusting TOT_EXP from weekly to monthly.")
    analysis_df['TOT_EXP_MONTHLY'] = analysis_df['TOT_EXP'] * WEEKS_PER_MONTH
    EXP_COL = 'TOT_EXP_MONTHLY'
else:
    print(f"Ratio = {ratio:.3f}. Units look monthly.")
    EXP_COL = 'TOT_EXP'


## 2. Sanity Checks: Asset Participation
**Why this step?** 
Before running complex regressions, we must ensure our baseline participation rates match known macroeconomic benchmarks (e.g., Badarinza AIDIS data). If these don't align, our sample weights or data cleaning might be flawed.


In [ ]:
def w_avg(group, val_col, wt_col):
    d = group.dropna(subset=[val_col, wt_col])
    if len(d) == 0: return np.nan
    vals = pd.to_numeric(d[val_col], errors='coerce')
    wts = pd.to_numeric(d[wt_col], errors='coerce')
    valid = vals.notna() & wts.notna()
    if not valid.any(): return np.nan
    return np.average(vals[valid], weights=wts[valid]) * 100

wave_df = analysis_df.drop_duplicates(subset=['HH_ID', 'WAVE_NO']).copy()

asset_rates = {
    'Real Estate': w_avg(wave_df, 'HAS_SAVING_IN_REAL_ESTATE', 'R_HH_WGT_W'),
    'Gold': w_avg(wave_df, 'HAS_SAVING_IN_GOLD', 'R_HH_WGT_W'),
    'Provident Fund (PF)': w_avg(wave_df, 'HAS_SAVING_IN_PF', 'R_HH_WGT_W'),
    'Mutual Funds': w_avg(wave_df, 'HAS_SAVING_IN_MF', 'R_HH_WGT_W'),
    'High Cost Debt': w_avg(wave_df, 'HIGH_COST_DEBT_FLAG', 'R_HH_WGT_W')
}

sanity_table = pd.DataFrame.from_dict(asset_rates, orient='index', columns=['CMIE Sample (%)'])
sanity_table['Benchmark Baseline (%)'] = ['~80-90%', '~80-90%', 'Low (~10-15%)', 'Very Low (<5%)', 'High']
print("=== Asset Composition Sanity Check ===")
print(sanity_table.round(2))


## 3. Core Regressions: Marginal Propensity to Consume (MPC)
**Why this step?** 
We estimate the MPC out of income. We use Weighted Least Squares (WLS) with cluster-robust standard errors grouped by `HH_ID` to account for repeated monthly observations of the same household. We also trim the top 1% to ensure extreme outliers aren't driving the slope.


In [ ]:
# Base Model
reg_df = analysis_df.dropna(subset=[EXP_COL, 'TOT_INC', 'R_HH_WGT_MS_INC', 'HH_ID']).copy()
model_base = sm.WLS(reg_df[EXP_COL], sm.add_constant(reg_df['TOT_INC']), weights=reg_df['R_HH_WGT_MS_INC']).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['HH_ID']}
)
mpc_base = model_base.params['TOT_INC']

# Trimmed Model (Top 1% dropped)
inc_p99 = reg_df['TOT_INC'].quantile(0.99)
trim_df = reg_df[reg_df['TOT_INC'] <= inc_p99].copy()
model_trim = sm.WLS(trim_df[EXP_COL], sm.add_constant(trim_df['TOT_INC']), weights=trim_df['R_HH_WGT_MS_INC']).fit(
    cov_type='cluster', cov_kwds={'groups': trim_df['HH_ID']}
)

print(f"Base MPC (Full Sample): {mpc_base:.3f}")
print(f"Trimmed MPC (Top 1% dropped): {model_trim.params['TOT_INC']:.3f}")


## 4. Permanent Income Hypothesis (PIH) Test
**Why this step?**
According to the PIH, households should consume heavily out of permanent income but save transitory income shocks. We decompose income into `PERM_INC` (household's own mean) and `TRANS_INC` (deviation from mean) and test if the coefficients differ significantly.


In [ ]:
# Calculate Permanent Income
hh_inc = analysis_df.groupby('HH_ID')['TOT_INC'].agg(PERM_INC='mean', N_MONTHS='count').reset_index()
hh_inc = hh_inc[hh_inc['N_MONTHS'] >= 8] # Require 8 months of data
pih_df = analysis_df.merge(hh_inc[['HH_ID', 'PERM_INC']], on='HH_ID', how='inner')
pih_df['TRANS_INC'] = pih_df['TOT_INC'] - pih_df['PERM_INC']

pih_reg = pih_df.dropna(subset=[EXP_COL, 'PERM_INC', 'TRANS_INC', 'R_HH_WGT_MS_INC', 'HH_ID']).copy()
model_pih = sm.WLS(pih_reg[EXP_COL], sm.add_constant(pih_reg[['PERM_INC', 'TRANS_INC']]), weights=pih_reg['R_HH_WGT_MS_INC']).fit(
    cov_type='cluster', cov_kwds={'groups': pih_reg['HH_ID']}
)

print("=== PIH Test Coefficients ===")
print(f"MPC out of Permanent Income:  {model_pih.params['PERM_INC']:.3f}")
print(f"MPC out of Transitory Income: {model_pih.params['TRANS_INC']:.3f}")
wald = model_pih.wald_test('PERM_INC = TRANS_INC', scalar=True)
print(f"Wald Test (H0: Perm == Trans): p-value = {wald.pvalue:.4g}")


## 5. Determinants of Co-Holding Puzzle
**Why this step?** 
We want to know what predicts holding High-Cost Debt and Illiquid Savings simultaneously. We regress `COHOLD_NUM` on income volatility (`INC_CV`) and formal employment (`FORMAL_NUM`).
We include two corrections:
1. **Selection Bias Correction**: Run without `INC_CV` to see if dropping households missing 8 months of income data skewed our formality finding.
2. **Conditional Correction**: Run predicting *High Cost Debt* conditioned only on households that already have *Illiquid Savings (PF)* to test if formality mechanically drives the co-holding flag just because formal workers automatically get PF.


In [ ]:
# Setup variables
cohold = wave_df.copy()
cohold['COHOLD_NUM'] = cohold['COHOLD_FLAG'].astype(float)
cohold['FORMAL_NUM'] = cohold['HAS_FORMAL_EMPLOYED_MEM'].astype(float)
cohold['STATE_WAVE'] = cohold['STATE_WAVE'].astype(str).fillna('Missing')
cohold['EDU_GROUP_WAVE'] = cohold['EDU_GROUP_WAVE'].astype(str).fillna('Missing')

# Model A: Base Model (with INC_CV, skewed sample)
reg_A = cohold.dropna(subset=['COHOLD_NUM', 'INC_CV', 'FORMAL_NUM', 'AGE_YRS_HOH', 'N_MEMBERS', 'R_HH_WGT_W']).copy()
X_A = sm.add_constant(pd.concat([reg_A[['INC_CV', 'FORMAL_NUM', 'AGE_YRS_HOH', 'N_MEMBERS']], pd.get_dummies(reg_A['STATE_WAVE'], drop_first=True)], axis=1)).astype(float)
model_A = sm.WLS(reg_A['COHOLD_NUM'], X_A, weights=reg_A['R_HH_WGT_W']).fit(cov_type='cluster', cov_kwds={'groups': reg_A['HH_ID']})

# Model B: Full Sample (No INC_CV, fixes selection bias)
reg_B = cohold.dropna(subset=['COHOLD_NUM', 'FORMAL_NUM', 'AGE_YRS_HOH', 'N_MEMBERS', 'R_HH_WGT_W']).copy()
X_B = sm.add_constant(pd.concat([reg_B[['FORMAL_NUM', 'AGE_YRS_HOH', 'N_MEMBERS']], pd.get_dummies(reg_B['STATE_WAVE'], drop_first=True)], axis=1)).astype(float)
model_B = sm.WLS(reg_B['COHOLD_NUM'], X_B, weights=reg_B['R_HH_WGT_W']).fit(cov_type='cluster', cov_kwds={'groups': reg_B['HH_ID']})

# Model C: Conditional Model (Fixes Mechanical Formality Bias)
illiquid_mask = cohold['HAS_SAVING_IN_PF'].isin([1, 1.0, 'Y', 'y', True])
reg_C = cohold[illiquid_mask].dropna(subset=['HIGH_COST_DEBT_FLAG', 'FORMAL_NUM', 'AGE_YRS_HOH', 'N_MEMBERS', 'R_HH_WGT_W']).copy()
reg_C['HIGH_COST_DEBT_NUM'] = reg_C['HIGH_COST_DEBT_FLAG'].astype(float)
X_C = sm.add_constant(pd.concat([reg_C[['FORMAL_NUM', 'AGE_YRS_HOH', 'N_MEMBERS']], pd.get_dummies(reg_C['STATE_WAVE'], drop_first=True)], axis=1)).astype(float)
model_C = sm.WLS(reg_C['HIGH_COST_DEBT_NUM'], X_C, weights=reg_C['R_HH_WGT_W']).fit(cov_type='cluster', cov_kwds={'groups': reg_C['HH_ID']})

print("=== Formality Coefficient Across Specifications ===")
print(f"Model A (Base, N={len(reg_A):,}):        {model_A.params['FORMAL_NUM']:.4f} (p={model_A.pvalues['FORMAL_NUM']:.4f})")
print(f"Model B (Full Sample, N={len(reg_B):,}): {model_B.params['FORMAL_NUM']:.4f} (p={model_B.pvalues['FORMAL_NUM']:.4f})")
print(f"Model C (Conditional, N={len(reg_C):,}): {model_C.params['FORMAL_NUM']:.4f} (p={model_C.pvalues['FORMAL_NUM']:.4f})")
if model_C.pvalues['FORMAL_NUM'] >= 0.05:
    print("-> Result: Formality significance disappears conditionally. The co-holding link was a mechanical artifact of PF mandates.")


## 6. Intention-Behavior Gap: PF vs Gold
**Why this step?**
If formality gives households better financial follow-through, we should see formal workers closing the intention-behavior gap on voluntary savings (Gold). If they only close the gap on Provident Funds (PF), it's just the mechanics of automated payroll deductions at work, not superior financial behavior.


In [ ]:
gap_col_pf = 'GAP_PF_JanApr_MayAug'
gap_col_gold = 'GAP_GOLD_JanApr_MayAug'

if gap_col_pf in wave_df.columns and gap_col_gold in wave_df.columns:
    # PF Model
    pf_df = wave_df[wave_df['WILL_SAVE_IN_PF'].isin([1, 1.0, 'Y', True])].dropna(subset=[gap_col_pf, 'FORMAL_NUM', 'R_HH_WGT_W']).copy()
    X_pf = sm.add_constant(pf_df['FORMAL_NUM']).astype(float)
    mod_pf = sm.WLS(pf_df[gap_col_pf].astype(float), X_pf, weights=pf_df['R_HH_WGT_W']).fit(cov_type='cluster', cov_kwds={'groups': pf_df['HH_ID']})
    
    # Gold Model
    gold_df = wave_df[wave_df['WILL_SAVE_IN_GOLD'].isin([1, 1.0, 'Y', True])].dropna(subset=[gap_col_gold, 'FORMAL_NUM', 'R_HH_WGT_W']).copy()
    X_gold = sm.add_constant(gold_df['FORMAL_NUM']).astype(float)
    mod_gold = sm.WLS(gold_df[gap_col_gold].astype(float), X_gold, weights=gold_df['R_HH_WGT_W']).fit(cov_type='cluster', cov_kwds={'groups': gold_df['HH_ID']})
    
    print("=== Intention-Behavior Gap (Effect of Formal Employment) ===")
    print(f"PF Gap:   {mod_pf.params['FORMAL_NUM']:.4f} (p={mod_pf.pvalues['FORMAL_NUM']:.4f})")
    print(f"Gold Gap: {mod_gold.params['FORMAL_NUM']:.4f} (p={mod_gold.pvalues['FORMAL_NUM']:.4f})")
    
    if mod_gold.pvalues['FORMAL_NUM'] >= 0.05:
        print("-> Conclusion: Formal workers ONLY have better follow-through on PF, proving it's an automated payroll artifact, not better behavior.")
else:
    print(f"Missing gap columns in dataset.")
